# Colab track — no Docker, no GPU, no live OLMo

This is the **demo** copy of `T1-11_dataset_and_frozen.ipynb`. Replies that would have come from OLMo/Phi are **scripted** (`[OLMo demo]` / `[Phi-4-mini demo]`). Days 2–3 already work without a model.

The live-hosting course is `tier1/` + Docker — see the [course README](https://github.com/justkrismanohar/fine-tuning-build/blob/main/README.md).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justkrismanohar/fine-tuning-build/blob/main/colab/day3/T1-11_dataset_and_frozen.ipynb)

**Runtime → CPU.** Then **Runtime → Run all**. First code cell clones the course (or finds it if you already have the folder).


In [ ]:
# [RUN] COLAB TRACK — find or clone the course, stay on demo replies
import os
import sys
import subprocess
from pathlib import Path

os.environ["CIVICDESK_COLAB"] = "1"
os.environ["CIVICDESK_MODE"] = "demo"

def _course_root() -> Path:
    markers = Path("tier1") / "day1" / "civicdesk_lab.py"
    for base in [Path("/content/fine-tuning-build"), Path.cwd(), *Path.cwd().resolve().parents]:
        if (base / markers).is_file():
            return base
    dest = Path("/content/fine-tuning-build")
    url = "https://github.com/justkrismanohar/fine-tuning-build.git"
    print("Cloning", url)
    subprocess.check_call(["git", "clone", "--depth", "1", url, str(dest)])
    return dest

ROOT = _course_root()
DAY = ROOT / "tier1" / "day3"
os.chdir(DAY)
if str(DAY) not in sys.path:
    sys.path.insert(0, str(DAY))
if "day3" == "day3":
    d2 = str(ROOT / "tier1" / "day2")
    if d2 not in sys.path:
        sys.path.insert(0, d2)
print("COLAB TRACK")
print(" course:", ROOT)
print(" cwd   :", Path.cwd())
print(" Live OLMo/Phi are OFF. Later chat cells use canned CivicDesk replies.")

# Markdown images in Colab resolve from /content, not cwd
import shutil
content = Path("/content") if Path("/content").is_dir() else Path.cwd()
for folder in ("images", "cards"):
    src = DAY / folder
    dest = content / folder
    if src.is_dir() and dest.resolve() != src.resolve():
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print(f" {folder} →", dest)


On Colab CPU, `TrainSession` and `load_adapter` skip with a clear error. Use RECORDED compares when present, or Runtime → T4 to train for real. Do not treat demo pre-flight as a measured baseline.


# T1-11 Prove the prompt fails, inspect data, train, see the difference

**Time:** 09:00–10:30 &nbsp;|&nbsp; **Evidence:** pre-flight JSON + your adapter folder under `adapters/`

**Outcomes**

- See that a careful prompt still fails on **this** machine (or say clearly that it did not).
- Inspect training rows without being given the hunt answers.
- Choose LoRA targets, train for real, save an adapter, load it, and compare base vs adapter live.


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** change only lines marked `# [CHANGE]`. Allowed values are named in the cell above.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path.

Git is not required. If you are stuck, load `completed/` for **this** notebook only and mark it as loaded from reference.


**Data ban:** CivicDesk fiction only. `heldout/cases.py` is a **test** set — do not train on it.


## Overnight readers — the chain

```
input text → tokenizer → embedding → transformer blocks → LM head → next token
```

A **LoRA adapter** is a small extra file on some of those blocks. The base stays frozen. Disable the file = rollback.

**Headline task:** résumé criteria (`meets` / `does_not_meet` / `human_review`). Silence is `human_review`, never a fail.

Full prompts: `heldout/prompts.py`. Full held-out cases: `heldout/cases.py`.


## Ministry link

| Cluster | Failure we prove |
|---|---|
| **Labour** | Silence on a CV labelled as failure |
| **Trade** | Security buried under a printer (Track A in T1-13) |
| **all PII** | Redaction hunt in section 5 |


## 0. Backend

What this cell does: pick the live model or demo. You should see the backend line and GPU note.


In [ ]:
# Colab track: demo/RECORDED. Free T4 may still run TrainSession if CUDA is free.
BACKEND = "demo"  # "olmo" | "phi" | "demo"

import civicdesk_ft as ft

cfg = ft.set_backend(BACKEND)


## 1. Where a LoRA would sit

What this cell does: print the linear layer names from the OLMo **config** (no weight download). Then count LoRA parameters for your chosen targets.


In [ ]:
# [RUN]
import civicdesk_ft as ft

ft.print_anatomy()


**What this means:** those names are the projections inside one transformer block. A LoRA of rank `r` on `q_proj` and `v_proj` is about one million trainable weights — not a new model.

# [CHANGE] fill from the printout. Try four attention names, then all seven.


In [ ]:
# [CHANGE] allowed: any of q_proj k_proj v_proj o_proj gate_proj up_proj down_proj
TARGETS = ["q_proj", "v_proj"]
RANK = 8  # try 8 or 16

import civicdesk_ft as ft

ft.lora_param_count(TARGETS, RANK)


## 2. Pre-flight — prove the problem on *this* machine

What this cell does: call the **base** model at temperature 0 on the frozen held-out set. It saves a dated JSON. It must **not** reuse briefing-note percentages. If a tripwire does not fire, that is a real result.


In [ ]:
# [CHANGE] drop a task if short on time. Default all four.
TASKS = ["criteria", "tickets", "redaction", "minutes"]

import civicdesk_ft as ft

record = ft.run_preflight(TASKS)


**What this means:** `measured: true` means LIVE, this session. `BACKEND=demo` or a down server writes an unmeasured file — then use RECORDED cards and say so.


## 3. Three failures, one per scenario

What this cell does: show input, gold, raw output, and a one-line why. LIVE only if **this exact case** is in today's pre-flight file.


In [ ]:
# [CHANGE] "R2-criteria" | "T1" | "M1"
SHOW = "R2-criteria"

import civicdesk_ft as ft

ft.show_failure(SHOW)


**Handback:** one sentence — what did the prompt ask for, and what did the model do? Then change `SHOW` and run the other two.


## 4. Criteria training rows

What this cell does: rebuild the seeded criteria set (same sha256 every time) and preview it. Evidence must be a résumé substring; `does_not_meet` never has empty evidence.


In [ ]:
# [RUN]
import runpy
from pathlib import Path
import civicdesk_ft as ft

runpy.run_path(str(Path("build_criteria_set.py")), run_name="__main__")
rows = ft.inspect_criteria_rows()


**Handback:** which of your T1-05 gold pairs does this set cover? Silence → `human_review` is the policy this file teaches.


## 5. Redaction hunt (data inspection score)

What this cell does: preview the inspect file **without** giving away which rows are planted. Full file: `../shared/resume_redaction/data/train_inspect_v1.json`.

This score grades **your** inspection, not the model. `None` is the answer when the problem is with the whole set.


In [ ]:
# [RUN]
import civicdesk_ft as ft

ft.inspect_training_rows()


In [ ]:
# [CHANGE] put the row id, or None for the set-level problem.
# Format example only (not an answer): "leak": "TR-07"
FOUND = {"leak": None, "inconsistent": None, "canary": None, "rewrite": None, "imbalance": None}

import civicdesk_ft as ft

ft.score_hunt(FOUND)


## 6. Train the criteria adapter, live

What this section does: real optimizer updates on the same TARGETS/RANK you set above. Rerunning `step` **continues** the session — watch the update number.

Needs a GPU with free memory. If it raises, read the message; Section 7 can use `ADAPTER = "prebuilt"` or a RECORDED run.


In [ ]:
# [RUN] initialize once. Reuses TARGETS and RANK from section 1.
import ft_train

try:
    session = ft_train.TrainSession(task="criteria", targets=TARGETS, rank=RANK)
    session.summary()
    session.show_example(0)
except RuntimeError as exc:
    session = None
    print("SKIP train:", exc)


**What to look for in `step`:** update number, train loss, and a nonzero weight-change on one LoRA matrix. Loss can go up as well as down.


In [ ]:
# [RUN] one update. Rerun this cell for update 2, 3, …
if session is not None:
    session.step(1)
else:
    print("No session — skipped.")


In [ ]:
# [CHANGE] how many more updates in one go (e.g. 20). On a T4, ~1–3 s each after load.
MORE = 20

if session is not None:
    session.step(MORE)
    session.history()
else:
    print("No session — skipped.")


In [ ]:
# [RUN] validation loss (no weight updates; not the same as task accuracy)
if session is not None:
    session.validate()
else:
    print("No session — skipped.")


In [ ]:
# [RUN] save your adapter. Lists the files created.
if session is not None:
    session.save("adapters/my_criteria_run")
else:
    print("No session — nothing saved. Use ADAPTER='prebuilt' in section 7 if the bake exists.")


**What the files are:** `adapter_config.json` + weight file = the LoRA; `training_report.json` = measured settings and losses; `logs/last_adapter.json` = handoff path for T1-12.


## 7. Load an adapter and see the difference, live

**What to look for:** label, evidence quoted from the résumé, valid JSON.

Three cases prove direction only. T1-12 scores all 18.


In [ ]:
# [CHANGE] your run if you trained enough; "prebuilt" for the instructor adapter on the same data.
ADAPTER = "adapters/my_criteria_run"  # or "prebuilt"

CASE_IDS = ["R2-C3", "R2-C1", "R5-C1"]
# R2-C3 = silence → should be human_review (not does_not_meet)
# R2-C1 = in-progress degree → human_review
# R5-C1 = clear meets (ACCA/CAT)

import ft_train

try:
    model = ft_train.load_adapter(ADAPTER, task="criteria")
    compare = ft_train.compare_live(model, "criteria", case_ids=CASE_IDS)
except Exception as exc:
    print("Live compare unavailable:", exc)
    try:
        compare = ft_train.load_recorded("criteria")
        print("Showing RECORDED rows for", CASE_IDS)
        for row in compare.get("rows") or []:
            if row["id"] in CASE_IDS:
                print("=" * 40, row["id"])
                print("base:", (row.get("base") or "")[:300])
                print("adapter:", (row.get("adapter") or "")[:300])
    except Exception as exc2:
        print("No recorded fallback either:", exc2)
        compare = None


**What you just saw — handback:** one sentence on what changed between base and adapter on the silence case.

**Handoff:** T1-12 loads the same `ADAPTER` value (also written to `logs/last_adapter.json`). Copy it.
